# Custom Envelope Functions

The `AnalyticKernel` models starspot appearance and disappearance with a **trapezoidal squared envelope**: spots rise linearly with timescale `tau`, hold a constant flux deficit for duration `lspot`, then decay symmetrically.

The kernel is computed as:

$$K(\tau) = R_\Gamma(\tau) \cdot \sigma_k^2 \sum_n |c_n|^2 \cos(n\,\omega_0\,\tau)$$

where $R_\Gamma(\tau) = \langle \Gamma(t)\,\Gamma(t+\tau)\rangle$ is the **autocorrelation of the squared envelope** $\Gamma(t) = \alpha^2(t)/\alpha_{\rm max}^2$.

To add a new envelope shape:
1. Register it with `params.register_envelope` so that `resolve_hparam`, `GPSolver`, and `MCMCSampler` all recognise the new keys automatically.
2. Subclass `AnalyticKernel` and override `R_Gamma(lag)` to evaluate the new autocorrelation function.

This notebook demonstrates the pattern with a **Gaussian envelope** and compares it to the default trapezoidal one.

## Deriving $R_\Gamma$ for a Gaussian envelope

For $\Gamma(t) = \exp(-t^2/2\sigma^2)$, the autocorrelation is the convolution of $\Gamma$ with itself:

$$R_\Gamma(\tau) = \int_{-\infty}^{\infty} \Gamma(t)\,\Gamma(t+\tau)\,dt$$

We work through this step by step in sympy.

In [1]:
from sympy import *

t, tau, sigma, omega = symbols('t tau sigma omega', real=True, positive=True)

# Define the envelope
Gamma = exp(-t**2 / (2 * sigma**2))
Gamma

exp(-t**2/(2*sigma**2))

In [2]:
# Write out the integrand Γ(t) · Γ(t + τ)
integrand = Gamma * Gamma.subs(t, t + tau)
integrand = powsimp(expand(integrand))
integrand

exp(-t**2/sigma**2 - t*tau/sigma**2 - tau**2/(2*sigma**2))

In [3]:
# Integrate the integrand over t from -∞ to ∞
R_Gamma = integrate(integrand, (t, -oo, oo)).simplify()
R_Gamma

sqrt(pi)*sigma*exp(-tau**2/(4*sigma**2))

In [4]:
# Take the Fourier transform of R_Gamma to get Γ²(ω):
Gamma2_omega = integrate(R_Gamma * exp(-I * omega * tau), (tau, -oo, oo)).simplify()
Gamma2_omega

2*pi*sigma**2*exp(-omega**2*sigma**2)

In [5]:
sqrt(Gamma2_omega)

sqrt(2)*sqrt(pi)*sigma*exp(-omega**2*sigma**2/2)

## Define the Gaussian envelope

For a Gaussian squared envelope $\Gamma(t) = \exp(-t^2/2\sigma^2)$, the autocorrelation derived above is:

$$R_\Gamma(\tau) = \sigma\sqrt{\pi}\,\exp\!\left(-\frac{\tau^2}{4\sigma^2}\right)$$

and its Fourier transform is:

$$\hat{\Gamma}(\omega) = \sigma\sqrt{2\pi}\,\exp\!\left(-\frac{\sigma^2\omega^2}{2}\right)$$

### Step 1 — Register the envelope

Call `register_envelope` once (at module load time or before first use) with an `EnvelopeSpec` that declares the signature key `sigma_gauss`. `resolve_hparam` will then recognise any hparam dict that contains this key and inject `tau = sigma_gauss` for modules that need a scalar timescale.

In [ ]:
import sys
sys.path.append("../..")

import numpy as np
import matplotlib.pyplot as plt
import jax.numpy as jnp

from src_jax.params import EnvelopeSpec, register_envelope, resolve_hparam
from src_jax.analytic_kernel import AnalyticKernel

def _resolve_gaussian(raw: dict) -> dict:
    """Inject tau = sigma_gauss for scalar-tau compatibility."""
    return {"tau": raw["sigma_gauss"], "sigma_gauss": raw["sigma_gauss"]}

try:
    register_envelope(EnvelopeSpec(
        name="gaussian",
        signature_keys=frozenset({"sigma_gauss"}),
        resolve=_resolve_gaussian,
        description="Gaussian decay: sigma_gauss sets the 1/e width [days]",
    ))
except ValueError:
    pass  # already registered if cell is re-run

### Step 2 — Subclass `AnalyticKernel` and override `R_Gamma`

The registered envelope handles validation and key injection. The subclass only needs to override `R_Gamma` (and optionally `compute_psd` if you want the analytic PSD).

In [ ]:
class GaussianEnvelopeKernel(AnalyticKernel):
    """
    AnalyticKernel with a Gaussian squared-envelope autocorrelation.

    The squared envelope is Γ(t) = exp(-t²/(2σ²)), giving
    R_Γ(τ) = sqrt(π) · σ · exp(-τ²/(4σ²)).

    Custom hparam key
    -----------------
    sigma_gauss : float  — Gaussian width [days]  (replaces tau)
    lspot       : not physically meaningful; set to 0.0

    All standard amplitude keys (sigma_k / nspot_rate+fspot+alpha_max)
    are accepted as usual.
    """

    def R_Gamma(self, lag):
        """R_Γ(τ) = sqrt(π) · σ · exp(-τ² / (4·σ²))"""
        sigma = self.hparam["sigma_gauss"]
        t = jnp.abs(jnp.asarray(lag, dtype=float).ravel())
        return jnp.sqrt(jnp.pi) * sigma * jnp.exp(-t**2 / (4 * sigma**2))

    def compute_psd(self, omega, lat_dist=None):
        """Analytic PSD using the Gaussian envelope Fourier transform."""
        from src_jax.analytic_kernel import _Gamma_hat  # reuse for trapezoidal check
        sigma = self.hparam["sigma_gauss"]
        omega = jnp.asarray(omega, dtype=float)

        def _gamma_hat(w):
            return jnp.sqrt(2 * jnp.pi) * sigma * jnp.exp(-sigma**2 * w**2 / 2)

        if lat_dist is None:
            lat_dist = lambda phi: 1.0

        import jax
        phi_min, phi_max = self.lat_range
        phi_grid = jnp.linspace(phi_min, phi_max, self.n_lat)
        dphi = phi_grid[1] - phi_grid[0]
        user_weights = jnp.array([lat_dist(float(p)) for p in phi_grid])
        norm = jnp.trapezoid(user_weights, phi_grid)

        def _psd_at_lat(phi):
            cn_sq = self.cn_squared(phi)
            w0 = self.omega0(phi)
            contrib = cn_sq[0] * _gamma_hat(omega) ** 2
            ns = jnp.arange(1, len(cn_sq))
            contribs = jax.vmap(lambda n: cn_sq[n] * (_gamma_hat(omega - n * w0) ** 2
                                                       + _gamma_hat(omega + n * w0) ** 2))(ns)
            return contrib + jnp.sum(contribs, axis=0)

        all_contribs = jax.vmap(_psd_at_lat)(phi_grid)
        psd = jnp.sum(user_weights[:, None] * all_contribs, axis=0) * dphi / norm
        return np.asarray(omega / (2 * jnp.pi)), np.asarray(psd * self.sigma_k ** 2)

## Compare envelope shapes

Look at the autocorrelation functions $R_\Gamma(\tau)$ for the two envelope models.

In [ ]:
hparam_trap = dict(
    peq=5.0, kappa=0.2, inc=np.pi/3,
    lspot=10.0, tau=3.0,
    sigma_k=0.01,
)

# Gaussian hparam: sigma_gauss identifies the registered envelope; lspot unused
hparam_gauss = dict(
    peq=5.0, kappa=0.2, inc=np.pi/3,
    lspot=0.0, sigma_gauss=3.0,
    sigma_k=0.01,
)

ak_trap  = AnalyticKernel(hparam_trap,  n_harmonics=3, n_lat=64)
ak_gauss = GaussianEnvelopeKernel(hparam_gauss, n_harmonics=3, n_lat=64)

tlags = np.linspace(0, 25, 500)

R_trap  = ak_trap.R_Gamma(tlags)
R_gauss = np.asarray(ak_gauss.R_Gamma(tlags))

plt.figure(figsize=(8, 4))
plt.plot(tlags, R_trap  / R_trap[0],  label="Trapezoidal (default)")
plt.plot(tlags, R_gauss / R_gauss[0], label="Gaussian", ls="--")
plt.axhline(0, color="gray", lw=0.8)
plt.xlabel("Lag [days]", fontsize=14)
plt.ylabel(r"$R_\Gamma(\tau)\,/\,R_\Gamma(0)$", fontsize=14)
plt.title("Envelope autocorrelation", fontsize=14)
plt.legend(fontsize=12)
plt.tight_layout()
plt.show()

## Compare GP kernels

Evaluate the full GP kernel $K(\tau)$ for both envelope models.

In [ ]:
K_trap  = ak_trap(tlags)
K_gauss = ak_gauss(tlags)

plt.figure(figsize=(8, 4))
plt.plot(tlags, K_trap  / K_trap[0],  label="Trapezoidal (default)")
plt.plot(tlags, K_gauss / K_gauss[0], label="Gaussian", ls="--")
for ii in range(1, 6):
    plt.axvline(ii * hparam_trap["peq"], color="gray", ls=":", alpha=0.6)
plt.axhline(0, color="gray", lw=0.8)
plt.xlabel("Lag [days]", fontsize=14)
plt.ylabel(r"$K(\tau)\,/\,K(0)$", fontsize=14)
plt.title("GP kernel", fontsize=14)
plt.legend(fontsize=12)
plt.tight_layout()
plt.show()

## Compare power spectral densities

In [ ]:
omega = np.linspace(0, 6 * np.pi / hparam_trap["peq"], 1000)

freq_trap,  psd_trap  = ak_trap.compute_psd(omega)
freq_gauss, psd_gauss = ak_gauss.compute_psd(omega)

plt.figure(figsize=(8, 4))
plt.semilogy(freq_trap,  psd_trap,  label="Trapezoidal (default)")
plt.semilogy(freq_gauss, psd_gauss, label="Gaussian", ls="--")
for ii in range(1, 4):
    plt.axvline(ii / hparam_trap["peq"], color="gray", ls=":", alpha=0.6,
                label=f"$f={ii}/P_{{eq}}$" if ii == 1 else None)
plt.xlabel("Frequency [cycles/day]", fontsize=14)
plt.ylabel("PSD", fontsize=14)
plt.title("Power spectral density", fontsize=14)
plt.legend(fontsize=12)
plt.tight_layout()
plt.show()